# imports

In [ ]:
import os
import random
import numpy as np
import torch
import torchaudio
import evaluate
from datasets import load_dataset, Dataset, Audio
from typing import Optional
from tqdm.auto import tqdm
from phonecodes import phonecodes

# Load Data

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    
DATA_PATH = PROJECT_ROOT / "data" / "fleurs" / 

In [ ]:
synth_folder = "/kaggle/input/ulster-irish-audio/wav_files/"
teanglann_folder = "/kaggle/input/ulster-irish-audio/wav_files/"
data_path = "/kaggle/input/ulster-wav-phon/irish_data_cleaned.csv"

In [ ]:
data_df = pd.read_csv(data_path)

# Inspect Data

# Prepare Data

In [ ]:
"""
build_l2_dataset.py
 
Builds an adversarially-concatenated training dataset for wav2vec2-based
mispronunciation detection, following the L2-GEN pairing strategy.
 
Each output instance concatenates a reference (native) and synthetic (L2)
pronunciation of the same word into a single audio array, with the
corresponding IPA strings joined as the label.
 
Expected input dataset schema
------------------------------
Column name       Type        Description
-----------       ----        -----------
word_id           str/int     Shared key linking ref & synthetic rows
source            str         "reference" or "synthetic"
audio             dict        HuggingFace Audio feature
                              {"array": np.ndarray, "sampling_rate": int}
ipa               str         IPA transcription string, e.g. "/θɪŋk/"
 
Output dataset schema
----------------------
word_id           str/int     Carried over from input
audio             dict        {"array": np.ndarray, "sampling_rate": 16000}
ipa               str         Concatenated IPA, e.g. "/θɪŋk/ /sɪŋk/"
order             str         "ref_first" or "synth_first" (for auditability)
"""
 
TARGET_SR = 16_000          # wav2vec2 expected sample rate
IPA_SEPARATOR = " "         # separator between the two IPA strings
RANDOM_SEED = 42            # to make reproduceable
 
 
def _to_mono_16k(audio_array: np.ndarray, source_sr: int) -> np.ndarray:
    """Resample and convert to mono float32 at TARGET_SR using torchaudio."""
    # Convert to float32 tensor — torchaudio expects (channels, time)
    waveform = torch.from_numpy(audio_array.astype(np.float32))
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)  # (1, time)
 
    # Convert stereo to mono if needed
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)  # (1, time)
 
    # Resample if needed
    if source_sr != TARGET_SR:
        resampler = torchaudio.transforms.Resample(
            orig_freq=source_sr, new_freq=TARGET_SR
        )
        waveform = resampler(waveform)
 
    # Return flat numpy array (time,) as expected downstream
    return waveform.squeeze(0).numpy()
 
 
def _concat_pair(
    ref_audio: np.ndarray,
    synth_audio: np.ndarray,
    ref_ipa: str,
    synth_ipa: str,
    rng: random.Random,
) -> dict:
    """
    Randomly order the two segments and concatenate audio + IPA.
    Returns a dict with keys: audio_array, ipa, order.
    """
    if rng.random() < 0.5:
        combined_audio = np.concatenate([ref_audio, synth_audio])
        combined_ipa = ref_ipa + IPA_SEPARATOR + synth_ipa
        order = "ref_first"
    else:
        combined_audio = np.concatenate([synth_audio, ref_audio])
        combined_ipa = synth_ipa + IPA_SEPARATOR + ref_ipa
        order = "synth_first"
 
    return {
        "audio_array": combined_audio,
        "ipa": combined_ipa,
        "order": order,
    }
 
 
def build_l2_training_set(
    dataset: Dataset,
    source_col: str = "source",
    ref_label: str = "reference",
    synth_label: str = "synthetic",
    word_id_col: str = "word_id",
    audio_col: str = "audio",
    ipa_col: str = "ipa",
    seed: Optional[int] = RANDOM_SEED,
) -> Dataset:
    """
    Build an adversarially-concatenated dataset from paired ref/synthetic rows.
 
    Parameters
    ----------
    dataset       : HuggingFace Dataset with the schema described at the top
    source_col    : column that distinguishes reference vs synthetic rows
    ref_label     : value in source_col that marks reference rows
    synth_label   : value in source_col that marks synthetic rows
    word_id_col   : column used to pair reference and synthetic rows
    audio_col     : HuggingFace Audio column name
    ipa_col       : IPA transcription column name
    seed          : random seed for order randomisation (None = non-deterministic)
 
    Returns
    -------
    HuggingFace Dataset with columns: word_id, audio, ipa, order
    """
    rng = random.Random(seed)
 
    # Cast audio column to ensure consistent decode behaviour
    dataset = dataset.cast_column(audio_col, Audio(sampling_rate=TARGET_SR))
 
    # Split into reference and synthetic subsets and index by word_id
    ref_rows = {}
    synth_rows = {}
 
    for row in dataset:
        wid = row[word_id_col]
        if row[source_col] == ref_label:
            ref_rows[wid] = row
        elif row[source_col] == synth_label:
            synth_rows[wid] = row
 
    # Find word IDs present in both subsets
    paired_ids = sorted(set(ref_rows.keys()) & set(synth_rows.keys()))
    unpaired = (set(ref_rows.keys()) | set(synth_rows.keys())) - set(paired_ids)
    if unpaired:
        print(f"[warn] {len(unpaired)} word_id(s) have no matching pair and will be skipped: "
              f"{list(unpaired)[:10]}{'...' if len(unpaired) > 10 else ''}")
 
    print(f"[info] Building {len(paired_ids)} concatenated instances...")
 
    # Build output rows
    out_word_ids = []
    out_audio_arrays = []
    out_sampling_rates = []
    out_ipas = []
    out_orders = []
 
    for wid in paired_ids:
        ref = ref_rows[wid]
        synth = synth_rows[wid]
 
        # Audio is already decoded to TARGET_SR by cast_column above,
        # but we still normalise dtype/channels to be safe
        ref_audio = _to_mono_16k(
            ref[audio_col]["array"], ref[audio_col]["sampling_rate"]
        )
        synth_audio = _to_mono_16k(
            synth[audio_col]["array"], synth[audio_col]["sampling_rate"]
        )
 
        result = _concat_pair(
            ref_audio=ref_audio,
            synth_audio=synth_audio,
            ref_ipa=ref[ipa_col],
            synth_ipa=synth[ipa_col],
            rng=rng,
        )
 
        out_word_ids.append(wid)
        out_audio_arrays.append(result["audio_array"])
        out_sampling_rates.append(TARGET_SR)
        out_ipas.append(result["ipa"])
        out_orders.append(result["order"])
 
    # Assemble output dataset
    out_dataset = Dataset.from_dict({
        word_id_col: out_word_ids,
        "audio": [
            {"array": arr, "sampling_rate": TARGET_SR}
            for arr in out_audio_arrays
        ],
        "ipa": out_ipas,
        "order": out_orders,
    })
 
    # Re-cast audio column so downstream code gets the HF Audio feature type
    out_dataset = out_dataset.cast_column("audio", Audio(sampling_rate=TARGET_SR))
 
    print(f"[info] Done. Output dataset has {len(out_dataset)} instances.")
    return out_dataset
 
 
# ---------------------------------------------------------------------------
# Quick smoke test — replace with your actual dataset load call
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    from datasets import load_dataset
 
    # Replace this with however you load your combined dataset, e.g.:
    # ds = load_dataset("path/to/your/dataset")["train"]
    ds = load_dataset("your_dataset_name_here")["train"]
 
    l2_ds = build_l2_training_set(ds)
 
    print(l2_ds)
    print(l2_ds[0]["ipa"])